In [1]:
from src.dataset.videoload import (
    GroupNormalize, GroupScale, GroupCenterCrop, 
    Stack, ToTorchFormatTensor
)
from src.tokenizer.multimodal_llama_tokenizer import MultimodalLlamaTokenizer
from src.dataset.it_dataset_mistral_new import ITTrainDataset_mistral_new
from src.train.trainer import collate_hd
# tokenizer = MultimodalLlamaTokenizer.from_pretrained("/mnt/petrelfs/wangchenting/Mistral-7B-Instruct-v0.2", local_files_only=True) 
from src.share_utils.special_token import special_tokens
tokenizer = MultimodalLlamaTokenizer.from_pretrained(
    "/mnt/petrelfs/share_data/likunchang/model/llm/Mistral-7B-Instruct-v0.2", 
    local_files_only=True,
    n_query=96,
    v_query=96,
    special_tokens = special_tokens, 
) 
from src.share_utils.my_easydict import MyEasyDict as edict
from torchvision import transforms
from src.dataset.create import get_train_transform, create_dataset, create_loader, create_sampler
from src.model.multimodalllm_pt import MultiModalLLM_PT
from torch.utils.data import DataLoader
from omegaconf import OmegaConf
from src.share_utils.py_config_utils import Config, eval_dict_leaf

anno_root_it = "/mnt/petrelfs/share_data/videointern/annotations/anno_instruction/videochat_new"
import torch
# torch.backends.cudnn.enabled = False


import argparse
def initialize_parser():
    parser = argparse.ArgumentParser()

    # Arguments
    parser.add_argument("--dset_name", type=str, default='charadesSTA', choices=["hl", 'charadesSTA'])
    parser.add_argument("--eval_split_name", type=str, default="val",
                        help="should match keys in video_duration_idx_path, must set for VCMR")
    parser.add_argument("--data_ratio", type=float, default=1.0,
                        help="how many training and eval data to use. 1.0: use all, 0.1: use 10%.")
    parser.add_argument("--results_root", type=str, default="results")
    parser.add_argument("--exp_id", type=str, default=None, help="id of this run, required at training")
    parser.add_argument("--max_es_cnt", type=int, default=200, help="number of epochs to early stop")
    parser.add_argument("--eval_epoch", type=int, default=5, help="inference epoch")
    parser.add_argument("--grad_clip", type=float, default=0.1, help="perform gradient clip, -1: disable")
    parser.add_argument("--eval_untrained", action="store_true", help="Evaluate on un-trained model")
    parser.add_argument("--resume_all", action="store_true", help="if --resume_all, load optimizer/scheduler/epoch as well")
    parser.add_argument("--start_epoch", type=int, default=None, help="if None, will be set automatically when using --resume_all")

    # Data config
    parser.add_argument("--max_q_l", type=int, default=-1)
    parser.add_argument("--max_v_l", type=int, default=-1)
    parser.add_argument("--clip_length", type=float, default=1.0)
    parser.add_argument("--max_windows", type=int, default=5)
    parser.add_argument("--train_path", type=str, default=None)
    parser.add_argument("--eval_path", type=str, default=None, help="Evaluating during training")
    parser.add_argument("--no_norm_vfeat", action="store_true", help="Do not normalize video feat")
    parser.add_argument("--no_norm_tfeat", action="store_true", help="Do not normalize text feat")
    parser.add_argument("--v_feat_dirs", type=str, nargs="+", help="video feature dirs")
    parser.add_argument("--t_feat_dir", type=str, help="text/query feature dir")
    parser.add_argument("--v_feat_dim", type=int, default=770, help="video feature dim")
    parser.add_argument("--t_feat_dim", type=int, default=4096, help="text/query feature dim")
    parser.add_argument("--ctx_mode", type=str, default="video_tef")

    # Model config
    parser.add_argument('--position_embedding', default='sine', type=str, choices=('sine', 'learned'),
                        help="Type of positional embedding to use")
    parser.add_argument('--enc_layers', default=3, type=int, help="Number of encoding layers in the transformer")
    parser.add_argument('--dec_layers', default=3, type=int, help="Number of decoding layers in the transformer")
    parser.add_argument('--t2v_layers', default=2, type=int, help="Number of decoding layers in the transformer")
    parser.add_argument('--sent_layers', default=1, type=int, help="Number of decoding layers in the transformer")
    parser.add_argument('--moment_layers', default=1, type=int, help="Number of decoding layers in the transformer")
    parser.add_argument('--dummy_layers', default=2, type=int, help="Number of encoding layers in the transformer")
    parser.add_argument('--dim_feedforward', default=1024, type=int, help="Intermediate size of the feedforward layers")
    parser.add_argument('--hidden_dim', default=256, type=int, help="Size of the embeddings")
    parser.add_argument('--input_dropout', default=0.5, type=float, help="Dropout applied in input")
    parser.add_argument('--dropout', default=0.1, type=float, help="Dropout applied in the transformer")
    parser.add_argument("--txt_drop_ratio", default=0, type=float, help="drop tokens from text input")
    parser.add_argument("--use_txt_pos", action="store_true", help="use position_embedding for text as well.")
    parser.add_argument('--nheads', default=8, type=int, help="Number of attention heads inside the transformer")
    parser.add_argument('--num_queries', default=10, type=int, help="Number of query slots")
    parser.add_argument('--num_dummies', default=45, type=int, help="Number of dummy tokens")
    parser.add_argument('--total_prompts', default=10, type=int, help="Number of query slots")
    parser.add_argument('--num_prompts', default=1, type=int, help="Number of dummy tokens")
    parser.add_argument('--pre_norm', action='store_true')
    parser.add_argument("--n_input_proj", type=int, default=2, help="#layers to encoder input")
    parser.add_argument("--contrastive_hdim", type=int, default=64, help="dim for contrastive embeddings")
    parser.add_argument("--temperature", type=float, default=0.07, help="temperature for contrastive loss")

    # Loss
    parser.add_argument("--saliency_margin", type=float, default=0.2)
    parser.add_argument('--no_aux_loss', dest='aux_loss', action='store_false', help="Disables auxiliary decoding losses")
    parser.add_argument("--span_loss_type", default="l1", type=str, choices=['l1', 'ce'],
                        help="Loss type for span")
    parser.add_argument("--contrastive_align_loss", action="store_true",
                        help="Disable contrastive alignment loss")
    parser.add_argument('--set_cost_span', default=10, type=float, help="L1 span coefficient in the matching cost")
    parser.add_argument('--set_cost_giou', default=1, type=float, help="GIoU span coefficient in the matching cost")
    parser.add_argument('--set_cost_class', default=4, type=float, help="Class coefficient in the matching cost")

    # Loss coefficients
    parser.add_argument("--lw_saliency", type=float, default=4, help="weight for saliency loss")
    parser.add_argument("--lw_wattn", type=float, default=1., help="weight for attention loss")
    parser.add_argument("--lw_ms_align", type=float, default=1., help="weight for multi-scale alignment loss")
    parser.add_argument("--lw_distill", type=float, default=1., help="weight for distillation loss")
    parser.add_argument('--span_loss_coef', default=10, type=float)
    parser.add_argument('--giou_loss_coef', default=1, type=float)
    parser.add_argument('--label_loss_coef', default=4, type=float)
    parser.add_argument('--eos_coef', default=0.1, type=float,
                        help="Relative classification weight of the no-object class")
    parser.add_argument("--contrastive_align_loss_coef", default=0.02, type=float)

    parser.add_argument("--no_sort_results", action="store_true", help="do not sort results")
    parser.add_argument("--max_before_nms", type=int, default=10)
    parser.add_argument("--max_after_nms", type=int, default=10)
    parser.add_argument("--conf_thd", type=float, default=0.0, help="only keep windows with conf >= conf_thd")
    parser.add_argument("--nms_thd", type=float, default=-1,
                        help="additional use of non-maximum suppression")

    return parser


from src.model.base_model import LLMConfig
config = LLMConfig.from_pretrained('/mnt/petrelfs/share_data/likunchang/model/llm/Mistral-7B-Instruct-v0.2', trust_remote_code=True)
print('aa')
cg_opt = initialize_parser()
model_cfg = eval_dict_leaf(Config.from_file('/mnt/petrelfs/yanziang/trackingllm/videoagent/configs/model/expert_umtl_f8_s2.py'))
# model_cfg = eval_dict_leaf(Config.from_file('/mnt/petrelfs/yanziang/trackingllm/videoagent/configs/model/i1b_qformer_mistral7b_f8.py'))
config.model_config = model_cfg.model
print('bb')
import sys
sys.argv = ['debug_mistral.ipynb']
config.cg_opt = cg_opt.parse_args()

config.model_tokenizer = tokenizer

from src.share_utils.distributed import get_world_size, get_rank, gather_together
from src.dataset.dataloader import MetaLoader
from src.dataset.create import get_media_types
from src.dataset.create import create_dataset


anno_root_it = "/mnt/petrelfs/share_data/videointern/annotations/anno_instruction/videochat_new"
import torch
# torch.backends.cudnn.enabled = False

model_cfg = eval_dict_leaf(Config.from_file('/mnt/petrelfs/yanziang/trackingllm/videoagent/configs/model/expert_umtl_f8_s2.py'))
# model_cfg = eval_dict_leaf(Config.from_file('/mnt/petrelfs/yanziang/trackingllm/videoagent/configs/model/i1b_qformer_mistral7b_f8.py'))
config.model_config = model_cfg.model
config.cg_opt = cg_opt.parse_args()
config.model_tokenizer = tokenizer

# path = '/mnt/petrelfs/share_data/yanziang/all'
path = '/mnt/petrelfs/share_data/yanziang/ckpt_last'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241007_115334/checkpoint-last'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241010_004436/checkpoint-120000'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241010_004436/checkpoint-last'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241013_101429/checkpoint-80000'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241016_115611/checkpoint-60000'
# path= '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241016_102651/checkpoint-100000'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241018_225600/checkpoint-100000'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241016_102651/checkpoint-60000'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241007_162703/checkpoint-last'
# path = '/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/scripts/pt/expert_tune/TVG_s2.sh_20241020_170132/checkpoint-last'
model = MultiModalLLM_PT.from_pretrained(path, config=config, local_files_only=True)
device = f'cuda:{0}'
model = model.to(device)

[2024-10-23 23:13:59,435] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/mnt/petrelfs/yanziang/.conda/envs/mllm/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-devel package with yum
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.4
 [WARNING]  using untested triton version (3.0.0), only 1.0.0 is known to be compatible


/mnt/petrelfs/yanziang/.conda/envs/mllm/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:49: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, weight, bias=None):
/mnt/petrelfs/yanziang/.conda/envs/mllm/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/mnt/petrelfs/yanziang/xformers/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/mnt/petrelfs/yanziang/xformers/xformers/ops/fmha/flash.py:338: FutureWarning: `torch.library.impl_abstract` was 

aa
bb


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

trainable params: 42,520,688 || all params: 7,284,310,128 || trainable%: 0.5837


/mnt/petrelfs/yanziang/.conda/envs/mllm/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.va

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of MultiModalLLM_PT were not initialized from the model checkpoint at /mnt/petrelfs/share_data/yanziang/ckpt_last and are newly initialized: ['sam.memory_encoder.fuser.layers.0.gamma', 'sam.memory_encoder.fuser.layers.1.gamma']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
def get_prompt(conv):
    ret = conv.system + conv.sep
    for role, message in conv.messages:
        if message:
            ret += role + " " + message + " " + conv.sep
        else:
            ret += role
    return ret


def get_prompt2(conv):
    ret = conv.system + conv.sep
    count = 0
    for role, message in conv.messages:
        count += 1
        if count == len(conv.messages):
            ret += role + " " + message
        else:
            if message:
                ret += role + " " + message + " " + conv.sep
            else:
                ret += role
    return ret


def get_context_emb(conv, model, img_list, answer_prompt=None, print_res=False):
    if answer_prompt:
        prompt = get_prompt2(conv)
    else:
        prompt = get_prompt(conv)
    if print_res:
        print(prompt)
    if '<VideoHere>' in prompt:
        prompt_segs = prompt.split('<VideoHere>')
    else:
        prompt_segs = prompt.split('<ImageHere>')
    assert len(prompt_segs) == len(img_list) + 1, "Unmatched numbers of image placeholders and images."
    with torch.no_grad():
        seg_tokens = [
            model.tokenizer(
                seg, return_tensors="pt", add_special_tokens=i == 0).to("cuda:0").input_ids
            # only add bos to the first seg
            for i, seg in enumerate(prompt_segs)
        ]
        seg_embs = [model.lm.base_model.model.model.embed_tokens(seg_t) for seg_t in seg_tokens]
    mixed_embs = [emb for pair in zip(seg_embs[:-1], img_list) for emb in pair] + [seg_embs[-1]]
    mixed_embs = torch.cat(mixed_embs, dim=1)
    return mixed_embs


def ask(text, conv):
    conv.messages.append([conv.roles[0], text])

from transformers import StoppingCriteria, StoppingCriteriaList

class StoppingCriteriaSub(StoppingCriteria):
    def __init__(self, stops=[], encounters=1):
        super().__init__()
        self.stops = stops
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
        for stop in self.stops:
            if torch.all((stop == input_ids[0][-len(stop):])).item():
                return True
        return False


from transformers import StoppingCriteria, StoppingCriteriaList

    
def answer(conv, model, img_list, do_sample=True, max_new_tokens=200, num_beams=1, min_length=1, top_p=0.9,
               repetition_penalty=1.0, length_penalty=1, temperature=1.0, answer_prompt=None, print_res=False):
    stop_words_ids = [
        torch.tensor([2]).to("cuda:0"),
        torch.tensor([29871, 2]).to("cuda:0")]  # '</s>' can be encoded in two different ways.
    stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(stops=stop_words_ids)])
    
    conv.messages.append([conv.roles[1], answer_prompt])
    embs = get_context_emb(conv, model, img_list, answer_prompt=answer_prompt, print_res=print_res)
    with torch.no_grad():
        outputs = model.lm.generate(
            inputs_embeds=embs,
            max_new_tokens=max_new_tokens,
            stopping_criteria=stopping_criteria,
            num_beams=num_beams,
            do_sample=do_sample,
            min_length=min_length,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            length_penalty=length_penalty,
            temperature=temperature,
        )
    output_token = outputs[0]
    if output_token[0] == 0:  # the model might output a unknow token <unk> at the beginning. remove it
            output_token = output_token[1:]
    if output_token[0] == 1:  # some users find that there is a start token <s> at the beginning. remove it
            output_token = output_token[1:]
    output_text = model.tokenizer.decode(output_token, add_special_tokens=False)
    output_text = output_text.split('</s>')[0]  # remove the stop sign </s>
#     output_text = output_text.split('[/INST]')[-1].strip()
    conv.messages[-1][1] = output_text + '</s>'
    return output_text, output_token.cpu().numpy()
import numpy as np
def get_index(num_frames, num_segments):
    seg_size = float(num_frames - 1) / num_segments
    start = int(seg_size / 2)
    offsets = np.array([
        start + int(np.round(seg_size * idx)) for idx in range(num_segments)
    ])
    return offsets

from torch.utils.data import Dataset
from torchvision.transforms.functional import InterpolationMode
from decord import VideoReader, cpu
import torchvision.transforms as T
from PIL import Image
def load_video(video_path, num_segments=8, return_msg=False, resolution=224):
    vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
    num_frames = len(vr)
    frame_indices = get_index(num_frames, num_segments)

    # transform
    crop_size = resolution
    scale_size = resolution
    input_mean = [0.48145466, 0.4578275, 0.40821073]
    input_std = [0.26862954, 0.26130258, 0.27577711]

    transform = T.Compose([
        GroupScale(int(scale_size), interpolation=InterpolationMode.BICUBIC),
        GroupCenterCrop(crop_size),
        Stack(),
        ToTorchFormatTensor(),
        GroupNormalize(input_mean, input_std) 
    ])

    images_group = list()
    for frame_index in frame_indices:
        img = Image.fromarray(vr[frame_index].numpy())
        images_group.append(img)
    torch_imgs = transform(images_group)
    if return_msg:
        fps = float(vr.get_avg_fps())
        sec = ", ".join([str(round(f / fps, 1)) for f in frame_indices])
        # " " should be added in the start and end
        msg = f"The video contains {len(frame_indices)} frames sampled at {sec} seconds."
        return torch_imgs, msg
    else:
        return torch_imgs

def get_sinusoid_encoding_table(n_position=784, d_hid=1024, cur_frame=8, ckpt_num_frame=4, pre_n_position=784): 
    ''' Sinusoid position encoding table ''' 
    # TODO: make it with torch instead of numpy 
    def get_position_angle_vec(position): 
        return [position / np.power(10000, 2 * (hid_j // 2) / d_hid) for hid_j in range(d_hid)] 
    
    # generate checkpoint position embedding
    sinusoid_table = np.array([get_position_angle_vec(pos_i) for pos_i in range(pre_n_position)]) 
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2]) # dim 2i 
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2]) # dim 2i+1 
    sinusoid_table = torch.tensor(sinusoid_table, dtype=torch.float, requires_grad=False).unsqueeze(0)
    
    print(f"n_position: {n_position}")
    print(f"pre_n_position: {pre_n_position}")
    
    if n_position != pre_n_position:
        T = ckpt_num_frame # checkpoint frame
        P = 14 # checkpoint size
        C = d_hid
        new_P = int((n_position // cur_frame) ** 0.5) # testing size
        if new_P != 14:
            print(f'Pretraining uses 14x14, but current version is {new_P}x{new_P}')
            print(f'Interpolate the position embedding')
            sinusoid_table = sinusoid_table.reshape(-1, T, P, P, C)
            sinusoid_table = sinusoid_table.reshape(-1, P, P, C).permute(0, 3, 1, 2)
            sinusoid_table = torch.nn.functional.interpolate(
                sinusoid_table, size=(new_P, new_P), mode='bicubic', align_corners=False)
            # BT, C, H, W -> BT, H, W, C ->  B, T, H, W, C
            sinusoid_table = sinusoid_table.permute(0, 2, 3, 1).reshape(-1, T, new_P, new_P, C)
            sinusoid_table = sinusoid_table.flatten(1, 3)  # B, THW, C
    
    if cur_frame != ckpt_num_frame:
        print(f'Pretraining uses 4 frames, but current frame is {cur_frame}')
        print(f'Interpolate the position embedding')
        T = ckpt_num_frame # checkpoint frame
        new_T = cur_frame # testing frame
        # interpolate
        P = int((n_position // cur_frame) ** 0.5) # testing size
        C = d_hid
        sinusoid_table = sinusoid_table.reshape(-1, T, P, P, C)
        sinusoid_table = sinusoid_table.permute(0, 2, 3, 4, 1).reshape(-1, C, T)  # BHW, C, T
        sinusoid_table = torch.nn.functional.interpolate(sinusoid_table, size=new_T, mode='linear')
        sinusoid_table = sinusoid_table.reshape(1, P, P, C, new_T).permute(0, 4, 1, 2, 3) # B, T, H, W, C
        sinusoid_table = sinusoid_table.flatten(1, 3)  # B, THW, C
        
    return sinusoid_table

vid_path = "/mnt/petrelfs/yanziang/output.mp4"
# vid_path = "./demo/example/jesse_dance.mp4"
# from src.utils.easydict import EasyDict
from src.share_utils.my_easydict import MyEasyDict as EasyDict

# num_frame = 8
num_frame = 16
# resolution = 384
resolution = 224
vid, msg = load_video(vid_path, num_segments=num_frame, return_msg=True, resolution=resolution)
new_pos_emb = get_sinusoid_encoding_table(n_position=(resolution//16)**2*num_frame, cur_frame=num_frame)
model.vision_encoder.encoder.pos_embed = new_pos_emb

print(msg)
    
# The model expects inputs of shape: T x C x H x W
TC, H, W = vid.shape
video = vid.reshape(1, TC//3, 3, H, W).to("cuda:0")
            # B, T, C, H, W = image.shape
video = video.permute(0, 2, 1, 3, 4)
img_list = []
with torch.no_grad():
    image_emb = model.encode_vision(video, "Watch the video and answer the question.")
    image_emb = model.project_up(image_emb).to(torch.bfloat16)
    # image_emb, _ = model.encode_img(video, "Watch the video and answer the question.")
#     image_emb, _ = model.encode_img(video, "")

img_list.append(image_emb)
chat = EasyDict({
    "system": "",
    "roles": ("[INST]", "[/INST]"),
    "messages": [],
    "sep": ""
})

chat.messages.append([chat.roles[0], "<Video><VideoHere></Video> [/INST]"])
ask("Describe the video in details.", chat)

llm_message = answer(conv=chat, model=model, do_sample=False, img_list=img_list, max_new_tokens=512,  print_res=True)[0]
print(llm_message)

n_position: 3136
pre_n_position: 784
Pretraining uses 4 frames, but current frame is 16
Interpolate the position embedding
The video contains 16 frames sampled at 1.6, 4.8, 8.0, 11.3, 14.5, 17.7, 20.9, 24.1, 27.4, 30.6, 33.8, 37.0, 40.3, 43.5, 46.7, 49.9 seconds.


/mnt/petrelfs/yanziang/.conda/envs/mllm/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:397: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] <Video><VideoHere></Video> [/INST] [INST] Describe the video in details. [/INST]
 The video shows a mouse running around on top of a stove in a kitchen. The mouse is seen moving around the stove, and at one point, it jumps off the stove and onto the counter. The kitchen has a white stove with a black top, and there is a white microwave on the counter next to the stove. The video also shows a white towel hanging on the oven door and a white cloth hanging on the oven handle. The video lasts for 8 seconds, and the mouse is the main focus of the video.


In [24]:
img_list = []
img_list.append(image_emb)
chat = EasyDict({
    "system": "",
    "roles": ("[INST]", "[/INST]"),
    "messages": [],
    "sep": ""
})

chat.messages.append([chat.roles[0], "<Video><VideoHere></Video> [/INST]"])

conv=chat
do_sample=False
img_list=img_list
max_new_tokens=512
print_res=True
num_beams=1
min_length=1
top_p=0.9

repetition_penalty=1.0
length_penalty=1
temperature=1.0
answer_prompt=None
stop_words_ids = [
    torch.tensor([2]).to("cuda:0"),
    torch.tensor([29871, 2]).to("cuda:0")]  # '</s>' can be encoded in two different ways.
stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(stops=stop_words_ids)])

conv.messages.append([conv.roles[1], answer_prompt])
embs = get_context_emb(conv, model, img_list, answer_prompt=answer_prompt, print_res=print_res)

print(embs.shape)
intermediate_outputs = []
intermediate_inputs = []
def hook_fn(module, input1, output):
    intermediate_inputs.append(input1)
    intermediate_outputs.append(output)


intermediate_outputs_emb = []
def hook_fn_emb(module, input, output):
    intermediate_outputs_emb.append(output)

# model.lm.base_model.model.model.embed_tokens.register_forward_hook(hook_fn_emb)

# for layer in model.lm.base_model.model.model.layers:
#     layer.register_forward_hook(hook_fn)


intermediate_outputs_layer1 = []
intermediate_inputs_layer1 = []
def hook_fn_layer1(module, input1, output):
    intermediate_inputs_layer1.append(input1)
    intermediate_outputs_layer1.append(output)

model.lm.base_model.model.model.layers[0].self_attn.register_forward_hook(hook_fn_layer1)

with torch.no_grad():
    outputs = model.lm.generate(
        inputs_embeds=embs,
        max_new_tokens=max_new_tokens,
        stopping_criteria=stopping_criteria,
        num_beams=num_beams,
        do_sample=do_sample,
        min_length=min_length,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        length_penalty=length_penalty,
        temperature=temperature,
    )





# # 打印中间层输出
# for i, output in enumerate(intermediate_outputs):
    
#     print(f"Layer {i} output shape: {output.shape}")




output_token = outputs[0]
print(len(output_token))
print(embs.shape)
if output_token[0] == 0:  # the model might output a unknow token <unk> at the beginning. remove it
        output_token = output_token[1:]
if output_token[0] == 1:  # some users find that there is a start token <s> at the beginning. remove it
        output_token = output_token[1:]
output_text = model.tokenizer.decode(output_token, add_special_tokens=False)
output_text = output_text.split('</s>')[0]  # remove the stop sign </s>
#     output_text = output_text.split('[/INST]')[-1].strip()
conv.messages[-1][1] = output_text + '</s>'
print(output_text)
print(output_token.cpu().numpy())
# return output_text, output_token.cpu().numpy()

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] <Video><VideoHere></Video> [/INST] [/INST]
torch.Size([1, 114, 4096])
90
torch.Size([1, 114, 4096])
 The video shows a mouse running across a white stove top with a pan on it. The mouse is small and brown, and it moves quickly across the surface of the stove. The stove has four burners, and the pan is black. The background of the video is a kitchen with a white microwave and a white refrigerator. The video is shot in a vertical orientation, and the mouse is the main focus of the video.
[  415  3798  4370   264 12733  3822  2673   264  3075 28479  1830   395
   264  3961   356   378 28723   415 12733   349  1741   304  9060 28725
   304   378 11662  4377  2673   272  5439   302   272 28479 28723   415
 28479   659  2308  5698   404 28725   304   272  3961   349  2687 28723
   415  5414   302   272  3798   349   264  6132   395   264  3075 15644
   671   458   304   264  3075 23641  1028 28723   415  3798   349  5180
   297   264 13519 18159 28725   304   272 12733   349   272  21

In [18]:
print(model.lm.base_model.model.model.layers[0])

MistralDecoderLayer(
  (self_attn): MistralSdpaAttention(
    (q_proj): lora.Linear(
      (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
      (lora_dropout): ModuleDict(
        (default): Dropout(p=0.1, inplace=False)
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=4096, out_features=16, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=16, out_features=4096, bias=False)
      )
      (lora_embedding_A): ParameterDict()
      (lora_embedding_B): ParameterDict()
    )
    (k_proj): lora.Linear(
      (base_layer): Linear(in_features=4096, out_features=1024, bias=False)
      (lora_dropout): ModuleDict(
        (default): Dropout(p=0.1, inplace=False)
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=4096, out_features=16, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=16, out_features=1024, bias=False)
      )
      (lora_embedding_A): Pa

In [23]:
intermediate_outputs_layer11 = intermediate_outputs_layer1
intermediate_inputs_layer11 = intermediate_inputs_layer1
print(intermediate_outputs_layer11[0][0])
print(intermediate_inputs_layer11)

tensor([[[ 2.6703e-03,  2.4605e-04,  1.1673e-03,  ...,  2.3346e-03,
           1.1215e-03,  1.7548e-03],
         [ 2.1362e-03, -3.5286e-05, -1.6212e-04,  ...,  4.9210e-04,
           4.1485e-05,  1.0681e-03],
         [ 1.2436e-03, -6.8188e-05, -1.2131e-03,  ...,  4.2439e-05,
          -5.7983e-04,  2.5177e-04],
         ...,
         [-6.8665e-04,  6.9427e-04, -2.0695e-04,  ..., -1.6632e-03,
          -9.3460e-04, -6.4087e-04],
         [-1.1826e-04,  7.6294e-04, -1.2970e-03,  ...,  1.1978e-03,
          -2.1820e-03, -7.4005e-04],
         [-4.3869e-04, -6.9427e-04,  1.1139e-03,  ..., -9.3842e-04,
           2.7847e-04, -2.0981e-04]]], device='cuda:0', dtype=torch.bfloat16)
[(), (), (), (), (), (), (), (), (), (), (), (), (), (), (), ()]


In [25]:
print(intermediate_outputs_layer1[0][0])
print(intermediate_inputs_layer1)

tensor([[[ 2.5177e-03, -6.8665e-05,  1.4038e-03,  ...,  2.6245e-03,
           1.1749e-03,  1.6937e-03],
         [ 1.9455e-03,  6.5327e-05, -1.9073e-05,  ...,  4.0817e-04,
           2.3365e-05,  9.1553e-04],
         [ 1.4496e-03,  1.4114e-04, -1.5488e-03,  ..., -3.5858e-04,
          -2.2316e-04,  3.6812e-04],
         ...,
         [-5.2643e-04,  6.4087e-04, -2.1553e-04,  ..., -1.6937e-03,
          -9.1553e-04, -8.2397e-04],
         [-4.9591e-05,  8.0872e-04, -1.2894e-03,  ...,  9.9182e-04,
          -2.2430e-03, -6.6757e-04],
         [-3.6621e-04, -7.9346e-04,  1.1139e-03,  ..., -7.8583e-04,
           2.0504e-04, -1.8311e-04]]], device='cuda:0', dtype=torch.bfloat16)
[(), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), (), ()

In [10]:
# for i, output in enumerate(intermediate_outputs):
#     print(f"Layer {i} output shape: {output[0].shape}")
print(len(intermediate_outputs))
# print(intermediate_outputs[1][0]) #不一样
intermediate_outputs1 = intermediate_outputs
intermediate_inputs1 = intermediate_inputs
print(intermediate_outputs1[0][0])
print(intermediate_inputs1[0][0])

2944
tensor([[[-1.7548e-03, -2.6703e-03, -1.4648e-01,  ...,  3.3569e-03,
           2.7161e-03,  3.1738e-03],
         [ 4.5471e-03, -1.2665e-03,  2.1210e-03,  ...,  2.7313e-03,
           4.0283e-03,  1.1749e-03],
         [ 3.5095e-03, -4.9744e-03, -6.2256e-03,  ..., -4.5471e-03,
           3.2043e-03, -6.4087e-04],
         ...,
         [ 2.4414e-04, -1.2207e-04,  5.8746e-04,  ..., -2.6550e-03,
          -9.4604e-03, -2.4567e-03],
         [ 6.3171e-03, -2.4109e-03, -2.1820e-03,  ..., -1.8768e-03,
          -1.3733e-04, -2.3346e-03],
         [-3.0212e-03, -2.7008e-03,  4.8218e-03,  ...,  8.5449e-04,
          -2.9602e-03,  4.1199e-03]]], device='cuda:0', dtype=torch.bfloat16)
tensor([[[-4.3640e-03, -1.0633e-04, -5.6152e-03,  ..., -5.0545e-05,
          -1.1520e-03,  1.5926e-04],
         [-4.3640e-03, -1.8954e-05, -1.7853e-03,  ...,  3.5858e-04,
           4.0588e-03,  4.5204e-04],
         [ 1.4496e-04,  5.0354e-04, -2.3499e-03,  ..., -2.5024e-03,
           3.2349e-03, -2.8229e-

In [16]:
print(len(intermediate_outputs))
print(len(intermediate_inputs))
print((intermediate_inputs[0][0] == intermediate_inputs1[0][0]).all()) #True
print((intermediate_outputs[0][0] == intermediate_outputs1[0][0]).all()) #False
print((intermediate_inputs[1][0] == intermediate_inputs1[1][0]).all()) #False
#model.lm.base_model.model.model.layers[0] 引入随机性

3136
3136
tensor(True, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')


In [16]:
print(model.lm.get_input_embeddings())

Embedding(32007, 4096)


In [17]:
aa = model.lm.get_input_embeddings()

In [18]:
bb = model.lm.get_input_embeddings()

In [19]:
aa == bb

True